# Smoke Test: All RAGAS Metrics

Runs all 4 metrics on 3 questions (~1-2 min) to verify the full pipeline before committing to the full run.

**Checks:**
- RAGAS service connectivity and model availability
- All metric names recognized (`answer_similarity`, `answer_correctness`, `context_precision`, `context_recall`)
- LLM responds and scores parse correctly
- Results saved in correct format for `ragas_results_comparison.ipynb`

**After this passes:** Run All on `ragas_eval_answer_metrics.ipynb` (which auto-triggers context metrics).

In [1]:
!pip install -q llama-stack-client==0.4.2 rich

In [2]:
import json
import os
import time
from datetime import datetime

from llama_stack_client import LlamaStackClient

RAGAS_URL = "http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321"
PROVIDER_ID_INLINE = "trustyai_ragas_inline"
SMOKE_SIZE = 3

ALL_METRICS = [
    "answer_similarity",
    "answer_correctness",
    "context_precision",
    "context_recall",
]


def compute_aggregated(score_result):
    agg = score_result.aggregated_results
    if agg is not None and not (isinstance(agg, dict) and None in agg.values()):
        if isinstance(agg, dict):
            vals = [v for v in agg.values() if v is not None]
            return vals[0] if len(vals) == 1 else agg
        return agg
    scores = []
    for row in score_result.score_rows:
        s = row.get("score")
        if s is not None and str(s) != "nan":
            scores.append(float(s))
    return round(sum(scores) / len(scores), 6) if scores else None

## 1. Connect and verify models

In [3]:
ragas_client = LlamaStackClient(base_url=RAGAS_URL)

ragas_models = ragas_client.models.list()
print("Models:")
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', '?')
    mtype = getattr(m, 'model_type', '?')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    print(f"  {mid} ({mtype})")

ragas_llm_model = None
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', None)
    mtype = getattr(m, 'model_type', '')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    if mtype == 'llm':
        ragas_llm_model = mid
        break

assert ragas_llm_model, "No LLM model found!"
print(f"\nLLM model: {ragas_llm_model}")

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


Models:
  vllm-embedding/qwen3-4b-embedding (embedding)
  vllm-inference/Gemma-3-27B-BF16-Distributed (llm)

LLM model: vllm-inference/Gemma-3-27B-BF16-Distributed


## 2. Load eval data (first 3 entries)

In [4]:
with open("eval_data_health_wallet.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

smoke_data = eval_data[:SMOKE_SIZE]
print(f"Full dataset: {len(eval_data)} entries")
print(f"Smoke test:   {SMOKE_SIZE} entries")
for i, d in enumerate(smoke_data):
    print(f"\n  Q{i+1}: {d['user_input'][:80]}")
    print(f"  A{i+1}: {d['response'][:80]}...")

Full dataset: 182 entries
Smoke test:   3 entries

  Q1: Kde nájdem Peňaženku zdravia? Je spoplatnená?
  A1: Peňaženku zdravia nájdete vo voľne dostupnej mobilnej aplikácii Všeobecnej zdrav...

  Q2: Prečo je Peňaženka zdravia viazaná na mobilnú aplikáciu?
  A2: Peňaženka zdravia je viazaná na mobilnú aplikáciu, pretože mobilná aplikácia pos...

  Q3: Na čerpanie príspevkov z Peňaženky zdravia musím mať mobilnú aplikáciu,
alebo mi
  A3: Na čerpanie príspevkov z Peňaženky zdravia stačí, ak máte zriadenú ePobočku....


## 3. Run all metrics

In [5]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
smoke_ds = f"hw_smoke_{ts}"
smoke_bm = f"hw_bench_smoke_{ts}"

ragas_client.beta.datasets.register(
    dataset_id=smoke_ds,
    purpose="eval/question-answer",
    source={"type": "rows", "rows": smoke_data},
    metadata={"provider_id": "localfs"},
)
ragas_client.alpha.benchmarks.register(
    benchmark_id=smoke_bm,
    dataset_id=smoke_ds,
    scoring_functions=ALL_METRICS,
    provider_id=PROVIDER_ID_INLINE,
)
job = ragas_client.alpha.eval.run_eval(
    benchmark_id=smoke_bm,
    benchmark_config={
        "eval_candidate": {
            "type": "model",
            "model": ragas_llm_model,
            "sampling_params": {"temperature": 0.1, "max_tokens": 1500},
        },
        "scoring_params": {},
    },
)
print(f"Job {job.job_id} submitted: {SMOKE_SIZE} questions x {len(ALL_METRICS)} metrics")

start = time.time()
while True:
    st = ragas_client.alpha.eval.jobs.status(benchmark_id=smoke_bm, job_id=job.job_id)
    elapsed = time.time() - start
    print(f"  [{elapsed:.0f}s] {st.status}")
    if st.status in ("completed", "failed"):
        break
    time.sleep(10)

assert st.status == "completed", f"FAILED after {elapsed:.0f}s — fix before running full eval!"
print(f"\nCompleted in {elapsed:.0f}s")

/tmp/ipykernel_2044/4157446683.py:5: DeprecationWarning: deprecated
  ragas_client.beta.datasets.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1beta/datasets "HTTP/1.1 200 OK"
/tmp/ipykernel_2044/4157446683.py:11: DeprecationWarning: deprecated
  ragas_client.alpha.benchmarks.register(
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0 "HTTP/1.1 200 OK"


Job 0 submitted: 3 questions x 4 metrics
  [0s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0 "HTTP/1.1 200 OK"


  [10s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0 "HTTP/1.1 200 OK"


...

  [170s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0 "HTTP/1.1 200 OK"


  [180s] in_progress


INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0 "HTTP/1.1 200 OK"


  [190s] completed

Completed in 190s


## 4. Verify scores

In [6]:
smoke_results = ragas_client.alpha.eval.jobs.retrieve(
    benchmark_id=smoke_bm, job_id=job.job_id
)

returned_metrics = list(smoke_results.scores.keys())
print(f"Metrics returned: {returned_metrics}")
print()

all_ok = True
for mn in ALL_METRICS:
    found = mn in smoke_results.scores
    if not found:
        # answer_similarity may come back as semantic_similarity
        found = (mn == "answer_similarity" and "semantic_similarity" in smoke_results.scores)
    if found:
        key = mn if mn in smoke_results.scores else "semantic_similarity"
        sr = smoke_results.scores[key]
        scores = [r.get("score") for r in sr.score_rows
                  if r.get("score") is not None and str(r.get("score")) != "nan"]
        skipped = len(sr.score_rows) - len(scores)
        avg = sum(scores) / len(scores) if scores else None
        status = "OK" if scores else "WARN: no valid scores"
        avg_str = f"{avg:.3f}" if avg is not None else "N/A"
        print(f"  {mn:25s}: avg={avg_str}, scored={len(scores)}/{SMOKE_SIZE}, "
              f"skipped={skipped} — {status}")
        if not scores:
            all_ok = False
    else:
        print(f"  {mn:25s}: MISSING — metric not recognized by provider!")
        all_ok = False

print()
if all_ok:
    print("ALL CHECKS PASSED. Safe to Run All on ragas_eval_answer_metrics.ipynb.")
else:
    print("SOME CHECKS FAILED. Fix the issues above before running the full evaluation.")

INFO:httpx:HTTP Request: GET http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321/v1alpha/eval/benchmarks/hw_bench_smoke_20260508_150419/jobs/0/result "HTTP/1.1 200 OK"


Metrics returned: ['semantic_similarity', 'answer_correctness', 'context_precision', 'context_recall']

  answer_similarity        : avg=0.957, scored=3/3, skipped=0 — OK
  answer_correctness       : avg=0.775, scored=3/3, skipped=0 — OK
  context_precision        : avg=0.972, scored=3/3, skipped=0 — OK
  context_recall           : avg=1.000, scored=3/3, skipped=0 — OK

ALL CHECKS PASSED. Safe to Run All on ragas_eval_answer_metrics.ipynb.


## 5. Save smoke results (for testing comparison notebook)

In [7]:
os.makedirs("results", exist_ok=True)

# Experiment config
experiment_config = {
    "experiment_id": "experiment1",
    "description": "Baseline: eurollm",
    "timestamp": datetime.now().isoformat(),
    "eval_model": ragas_llm_model,
    "num_questions": SMOKE_SIZE,
}
with open("results/experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)
print("Saved results/experiment_config.json")

# Answer metrics (individual files)
for mn in ["answer_similarity", "answer_correctness"]:
    key = mn if mn in smoke_results.scores else ("semantic_similarity" if mn == "answer_similarity" else None)
    if key and key in smoke_results.scores:
        sr = smoke_results.scores[key]
        m_result = {"metric": mn, "aggregated": compute_aggregated(sr),
                    "num_scored": 0, "num_skipped": 0, "per_question": []}
        for i, gen in enumerate(smoke_results.generations):
            score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None
            is_valid = score is not None and str(score) != "nan"
            if is_valid:
                m_result["num_scored"] += 1
            else:
                m_result["num_skipped"] += 1
            m_result["per_question"].append({
                "question": gen["user_input"],
                "score": round(float(score), 4) if is_valid else None,
                "response": smoke_data[i]["response"] if i < len(smoke_data) else "",
                "reference": smoke_data[i]["reference"] if i < len(smoke_data) else "",
            })
        with open(f"results/{mn}.json", "w", encoding="utf-8") as f:
            json.dump(m_result, f, ensure_ascii=False, indent=2)
        print(f"Saved results/{mn}.json (mean={m_result['aggregated']})")

# Context metrics (bundled file)
ctx_result = {}
for mn in ["context_precision", "context_recall"]:
    if mn in smoke_results.scores:
        sr = smoke_results.scores[mn]
        m_data = {"metric": mn, "aggregated": compute_aggregated(sr),
                  "num_scored": 0, "num_skipped": 0, "per_question": []}
        for i, gen in enumerate(smoke_results.generations):
            score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None
            is_valid = score is not None and str(score) != "nan"
            if is_valid:
                m_data["num_scored"] += 1
            else:
                m_data["num_skipped"] += 1
            m_data["per_question"].append({
                "question": gen["user_input"],
                "score": round(float(score), 4) if is_valid else None,
            })
        ctx_result[mn] = m_data

if ctx_result:
    with open("results/context_metrics.json", "w", encoding="utf-8") as f:
        json.dump(ctx_result, f, ensure_ascii=False, indent=2)
    print("Saved results/context_metrics.json")

print("\nSmoke results saved. You can now test ragas_results_comparison.ipynb.")
print("These will be overwritten when the full evaluation runs.")

Saved results/experiment_config.json
Saved results/answer_similarity.json (mean=0.9571760082186476)
Saved results/answer_correctness.json (mean=0.7747191032692368)
Saved results/context_metrics.json

Smoke results saved. You can now test ragas_results_comparison.ipynb.
These will be overwritten when the full evaluation runs.
